# Introduction to PyTorch
PyTorch is one of the most popular frameworks in the field of Deep Learning. Initially developed by Meta AI, and now it is a part of Linux Foundation. 

### 1) PyTorch Tensor
Tensors are the fundemental data structure in PyTorch, which is similar to an array or a matrix. It can support many mathematical operations and is the building block of neural networks. They can be created from Python list or NumPy arrays.


In [1]:
# import the PyTorch module
import torch

# define a tensor from a Python list
my_tensor = torch.tensor([[1, 2, 3], [4, 5, 6]])
print("Tensor:", my_tensor)
print("Shape:", my_tensor.shape)
print("Data type:", my_tensor.dtype)
print("Device:", my_tensor.device)

Tensor: tensor([[1, 2, 3],
        [4, 5, 6]])
Shape: torch.Size([2, 3])
Data type: torch.int64
Device: cpu


Mathematical operations can be done on two tensors as operands if they are compatible (same shape with aligned dimensions).

In [2]:
# define two operands
a = torch.tensor([1, 2, 3])
b = torch.tensor([4, 5, 6])

# perform addition
c = a + b
print("Result of addition:", c)

# perform subtraction
d = a - b
print("Result of subtraction:", d)

# perform element-wise multiplication
e = a * b
print("Result of element-wise multiplication:", e)

# perform matrix multiplication (with matmul)
f = torch.matmul(a, b)
print("Result of matrix multiplication:", f)

# perform matrix multiplication (with @ operator)
g = a @ b
print("Result of matrix multiplication with @ operator:", g)

Result of addition: tensor([5, 7, 9])
Result of subtraction: tensor([-3, -3, -3])
Result of element-wise multiplication: tensor([ 4, 10, 18])
Result of matrix multiplication: tensor(32)
Result of matrix multiplication with @ operator: tensor(32)


### 2) Designing a Neural Network
A neural network consists of input, hidden and output layers where the input layer contains the dataset features, the output layer contains the predictions (classes) and the hidden layers lie in between. A layer is *fully connected* when each neuron links to all neurons in the previous layer. A neuron in a linear layer performs a linear operation using all neurons from the previous layer, hence has *N+1* parameters (*N* from inputs and *1* from the bias). More hidden layers means more parameters and higher model capacity. However, too many parameters can lead to long training times or overfitting.

- **NN with no hidden layers:**

In [3]:
# define a nn with no hidden layers (fully connected, linear)
import torch.nn as nn

# create input tensor with 3 features
input_tensor = torch.tensor([1.0, 2.0, 3.0])

# define a linear layer with 3 input features and 2 output features
linear_layer = nn.Linear(in_features=3, out_features=2)

# pass the input tensor through the linear layer
output_tensor = linear_layer(input_tensor)

# print the output tensor, weights, and bias
print("Output tensor:", output_tensor)
print("Weights:", linear_layer.weight)
print("Bias:", linear_layer.bias)

Output tensor: tensor([-1.8865, -0.5494], grad_fn=<ViewBackward0>)
Weights: Parameter containing:
tensor([[-0.5354, -0.0886, -0.4575],
        [-0.4349, -0.4491,  0.3100]], requires_grad=True)
Bias: Parameter containing:
tensor([ 0.1984, -0.1463], requires_grad=True)


**NN with hidden layers:**

In [4]:
# create a fully connected nn with 3 linear layers and 2 hidden layers
model = nn.Sequential(
    nn.Linear(in_features=8, out_features=4),  # first hidden layer
    nn.Linear(in_features=4, out_features=2)   # output layer
)

input_tensor = torch.Tensor([[2, 3, 6, 7, 9, 3, 2, 1]])
output_tensor = model(input_tensor)
print("Output tensor:", output_tensor)

Output tensor: tensor([[ 2.5130, -1.0563]], grad_fn=<AddmmBackward0>)


**NN with activation functions:**
Activation functions add *non-linearity* to neural networks and make nns learn more complex relationships.
- Sigmoid: for binary classification
- Softmax: for multi-class classification


In [5]:
# create a binary classification model with sigmoid activation function at output layer
# Note: A nn with a single layer followed by a sigmoid activation function is similar to a logistic regression model
model_sigmoid = nn.Sequential( 
    nn.Linear(in_features=8, out_features=4),  # first linear layer
    nn.Linear(in_features=4, out_features=1),   # second linear layer
    nn.Sigmoid()  # activation function
)

# create a 3 class multi-class classification model with softmax activation function at output layer
model_softmax = nn.Sequential( 
    nn.Linear(in_features=8, out_features=4),  # first linear layer
    nn.Linear(in_features=4, out_features=3),   # second linear layer
    nn.Softmax(dim=-1)  # activation function
)

**NN model parameters:**

In [6]:
import torch.optim as optim

# create a sequential model
model = nn.Sequential(
    nn.Linear(in_features=16, out_features=8),  # first linear layer
    nn.Linear(in_features=8, out_features=2)   # second linear layer
)

print("Model architecture:", model)
print("Model parameters:")

total_params = 0
for name, param in model.named_parameters():
    print(f"{name}: {param.shape}")
    total_params += param.numel()

# model capacity = (16+1) * 8 + (8+1) * 2 = 154 learnable parameters
print(f"Total learnable parameters: {total_params}")

sample = torch.Tensor([[2, 3, 6, 7, 9, 3, 2, 1, 2, 3, 6, 7, 9, 3, 2, 1]])
target = torch.Tensor([[1, 9]])  # example target for binary classification
prediction = model(sample)

# create the optimizer
optimizer = optim.SGD(model.parameters(), lr=0.001)

# calculate the loss and gradients
criterion = nn.CrossEntropyLoss()
loss = criterion(prediction, target)
loss.backward()

# update the model's parameters using the optimizer
optimizer.step()

# access the gradients of the weight of the second linear layer (before backward pass)
grads1 = model[1].weight.grad
print("Gradient of the weight of the second layer:", grads1)

Model architecture: Sequential(
  (0): Linear(in_features=16, out_features=8, bias=True)
  (1): Linear(in_features=8, out_features=2, bias=True)
)
Model parameters:
0.weight: torch.Size([8, 16])
0.bias: torch.Size([8])
1.weight: torch.Size([2, 8])
1.bias: torch.Size([2])
Total learnable parameters: 154
Gradient of the weight of the second layer: tensor([[ 28.1694,  -3.8718,  48.2743, -25.9604,  35.5168,  -2.5834,   1.7177,
         -27.8656],
        [-28.1694,   3.8718, -48.2743,  25.9604, -35.5168,   2.5834,  -1.7177,
          27.8656]])


### 3) Training a Neural Network with PyTorch

**Loading Data:** 
Structuring data into a dataset is one of the first steps in training a PyTorch neural network. TensorDataset simplifies structuring data by converting NumPy arrays into a format PyTorch can use. After structuring the data, DataLoader class is essential for efficiently handling large datasets. It speeds up training, optimizes memory usage, and stabilizes gradient updates, making deep learning models more effective.

In [7]:
import torch
import pandas as pd
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

# create 'animals' as a pandas DataFrame containing the dataset
animals = pd.DataFrame({
    'animal_name': ['sparrow', 'eagle', 'cat', 'dog', 'lion', 'lizard'],
    'hair': [0, 0, 1, 1, 1, 0],
    'feathers': [1, 1, 0, 0, 0, 0],
    'eggs': [1, 1, 0, 0, 0, 1],
    'milk': [0, 0, 1, 1, 1, 0],
    'predator': [0, 1, 1, 0, 1, 1],
    'legs': [2, 2, 4, 4, 4, 4],
    'tail': [1, 1, 1, 1, 1, 1],
    'type': [0, 0, 1, 1, 1, 2]  # bird (0), mammal (1), reptile (2)
})

# extract features and target variable from the DataFrame
X = animals.iloc[:, 1:-1].to_numpy()  
y = animals.iloc[:, -1].to_numpy()

# create a tensor dataset
# CrossEntropyLoss expects LongTensor (int64) for labels, and float32 for features
dataset = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))

# print the first sample
input_sample, label_sample = dataset[0]
print('Input sample:', input_sample)
print('Label sample:', label_sample)

# create a DataLoader
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

# iterate over the dataloader
for batch_inputs, batch_labels in dataloader:
    print('batch_inputs:', batch_inputs)
    print('batch_labels:', batch_labels)

Input sample: tensor([0., 1., 1., 0., 0., 2., 1.])
Label sample: tensor(0)
batch_inputs: tensor([[0., 1., 1., 0., 1., 2., 1.],
        [0., 0., 1., 0., 1., 4., 1.]])
batch_labels: tensor([0, 2])
batch_inputs: tensor([[1., 0., 0., 1., 1., 4., 1.],
        [1., 0., 0., 1., 1., 4., 1.]])
batch_labels: tensor([1, 1])
batch_inputs: tensor([[0., 1., 1., 0., 0., 2., 1.],
        [1., 0., 0., 1., 0., 4., 1.]])
batch_labels: tensor([0, 1])


**Training Loop:** Looping through the entire dataset once is called an epoch. In practice, we often loop through the dataset multiple times (multiple epochs) to allow the model to learn better representations of the data. In ```scikit-learn```, the training loop is wrapped in the ```.fit()``` method, while in PyTorch, it's set up manually. While this adds flexibility, it requires a custom implementation.

In [8]:
# define a simple model for multi-class classification
model = nn.Sequential(
    nn.Linear(in_features=7, out_features=8),  # first linear layer
    nn.ReLU(),  # activation function
    nn.Linear(in_features=8, out_features=3),   # second linear layer
)

# define the loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

num_epochs = 30

# loop over the number of epochs and the dataloader
for i in range(num_epochs):
  for data in dataloader:
    # set the gradients to zero
    optimizer.zero_grad()

    # run a forward pass
    feature, target = data
    prediction = model(feature)

    # compute the loss
    loss = criterion(prediction, target)

    # compute the gradients
    loss.backward()

    # update the model parameters
    optimizer.step()

# show evaluation results
model.eval()
with torch.no_grad():
    for feature, target in dataloader:
        prediction = model(feature)
        predicted_classes = prediction.argmax(dim=1)
        for p, t in zip(predicted_classes, target):
            print(f'Ground truth type: {t.item()}. Predicted type: {p.item()}.')

Ground truth type: 1. Predicted type: 1.
Ground truth type: 0. Predicted type: 0.
Ground truth type: 0. Predicted type: 0.
Ground truth type: 2. Predicted type: 2.
Ground truth type: 1. Predicted type: 1.
Ground truth type: 1. Predicted type: 1.


The *learning rate* (which is set as 0.01 above) controls the step size and setting a rate too high causes poor performance, while setting it too low results in slower training. Typical range for *learning rate* is 0.01 and 0.0001. On the other hand, *momentum* controls the inertia and helps escaping the local minimum. If it is set too small, optimizer gets stuck. The typical range for *momentum* is 0.85 to 0.99.

### 3) Evaluating and Improving Models

**Layer Initialization:** The initialization of the weights of a neural network has been researched for many years. When training a neural network, the initialization of the weights has a direct impact on the final performance of the network.

In [9]:
layer = nn.Linear(in_features=64, out_features=128)
nn.init.uniform_(layer.weight)
print("Layer weights after uniform initialization (min, max):", layer.weight.min(), layer.weight.max())

Layer weights after uniform initialization (min, max): tensor(0.0002, grad_fn=<MinBackward1>) tensor(0.9999, grad_fn=<MaxBackward1>)


**Transfer Learning:** Transfer learning takes a model that was trained on a first task and reuses it for a second task. When the tasks are similar, transfer learning may be done with fine-tuning by loading weights from the previously trained model, but by training the model with a smaller learning rate. It is also possible to only train a part of the pre-trained network by freezing the unused layers. A rule of thumb is to freeze the early layers of the network and fine-tune the layers closer to the output layer.

In [10]:
# save and load the layer
layer = nn.Linear(in_features=64, out_features=128)
torch.save(layer, 'layer.pth')
new_layer = torch.load('layer.pth', weights_only=False)

model = nn.Sequential(
    layer,  # first linear layer
    nn.Linear(in_features=8, out_features=2)   # second linear layer
)

for name, param in model.named_parameters():
    # Check for first layer's weight
    if name == '0.weight':
        # Freeze this weight
        param.requires_grad = False

**Evaluating Performance:** A dataset is typically split into three subsets.
- *Training set (80-90%)*: Adjusts model parameters such as weights and biases.
- *Validation set (10-20%)*: Tunes hyperparameters like learning rate and momentum.
- *Test set (5-10%)*: Evaluates final model performance

For model evauation, *loss* and *accuracy* is being tracked during training and validation. If training loss keeps decreasing while the validation loss stars to rise, the model overfits. This means the model is learning the training data too well and won't perform well on new data. While loss tells how a model is learning, accuracy reflects how accurately it makes predictions.


In [12]:
import torchmetrics

# define a simple model for multi-class classification
model = nn.Sequential(
    nn.Linear(in_features=7, out_features=8),  # first linear layer
    nn.ReLU(),  # activation function
    nn.Linear(in_features=8, out_features=3),   # second linear layer
)

# define the loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# define the accuracy metric
metric = torchmetrics.Accuracy(task='multiclass', num_classes=3)

num_epochs = 30

# loop over the number of epochs and the dataloader
for epoch in range(num_epochs):
    # initialize the training loss for this epoch
    train_loss = 0.0
    
    for data in dataloader:
        # clear gradients BEFORE the forward pass (this is a common practice to avoid accumulating gradients from previous iterations)
        optimizer.zero_grad()

        # forward pass
        feature, target = data
        prediction = model(feature)

        # compute batch accuracy
        metric.update(prediction, target)

        # compute the loss
        loss = criterion(prediction, target)

        # backpropagation
        loss.backward() # compute gradients
        optimizer.step() # update weights

        # calculate and sum the loss
        train_loss += loss.item() # .item() gets the scalar value of the loss tensor
    epoch_loss = train_loss / len(dataloader)

    # compute epoch accuracy
    epoch_accuracy = metric.compute()

    # reset the metric for the next epoch
    metric.reset()

    # use epoch_loss to track progress
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')

Epoch [1/30], Loss: 1.0203
Epoch [2/30], Loss: 0.9670
Epoch [3/30], Loss: 0.9241
Epoch [4/30], Loss: 0.8764
Epoch [5/30], Loss: 0.8390
Epoch [6/30], Loss: 0.7997
Epoch [7/30], Loss: 0.7575
Epoch [8/30], Loss: 0.7121
Epoch [9/30], Loss: 0.6697
Epoch [10/30], Loss: 0.6254
Epoch [11/30], Loss: 0.5830
Epoch [12/30], Loss: 0.5434
Epoch [13/30], Loss: 0.5023
Epoch [14/30], Loss: 0.4620
Epoch [15/30], Loss: 0.4255
Epoch [16/30], Loss: 0.3956
Epoch [17/30], Loss: 0.3649
Epoch [18/30], Loss: 0.3365
Epoch [19/30], Loss: 0.3185
Epoch [20/30], Loss: 0.2921
Epoch [21/30], Loss: 0.2703
Epoch [22/30], Loss: 0.2533
Epoch [23/30], Loss: 0.2342
Epoch [24/30], Loss: 0.2166
Epoch [25/30], Loss: 0.2039
Epoch [26/30], Loss: 0.1890
Epoch [27/30], Loss: 0.1762
Epoch [28/30], Loss: 0.1646
Epoch [29/30], Loss: 0.1553
Epoch [30/30], Loss: 0.1434


In [13]:
 # create validation set
animals_val = pd.DataFrame({
    'animal_name': ['parrot', 'horse'],
    'hair': [0, 1],
    'feathers': [1, 0],
    'eggs': [1, 0],
    'milk': [0, 1],
    'predator': [0, 0],
    'legs': [2, 4],
    'tail': [1, 1],
    'type': [0, 1]  # bird (0), mammal (1), reptile (2)
})

# extract features and target variable from the DataFrame
X_val = animals_val.iloc[:, 1:-1].to_numpy()  
y_val = animals_val.iloc[:, -1].to_numpy()

# create a DataLoader
validationloader = DataLoader(TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.long)), batch_size=2, shuffle=True)

validation_loss = 0.0
model.eval()  # set the model to evaluation mode
with torch.no_grad():  # disable gradient calculation for validation
    for feature, target in validationloader:
        prediction = model(feature)
        
        # get the predicted class by finding the index of the maximum value in the prediction tensor
        predicted_classes = prediction.argmax(dim=1)
        for p, t in zip(predicted_classes, target):
            print(f'Ground truth type: {t.item()}. Predicted type: {p.item()}.')

        loss = criterion(prediction, target)
        validation_loss += loss.item()  # sum the loss over the batch

validation_loss /= len(validationloader)  # compute the average loss
print(f'Validation Loss: {validation_loss:.4f}')
        
model.train()  # set the model back to training mode

Ground truth type: 0. Predicted type: 0.
Ground truth type: 1. Predicted type: 1.
Validation Loss: 0.0778


Sequential(
  (0): Linear(in_features=7, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=3, bias=True)
)

Use argmax if you are manually comparing raw model outputs to integer labels using standard PyTorch operations (like pred.argmax(dim=-1) == target). Skip argmax when passing raw outputs directly into torch.nn.CrossEntropyLoss() or torchmetrics.Accuracy(task='multiclass'). They both want the raw scores.